# Ising coupling -- Figure 12

$\chi$ at $E_g + 0.1$ eV (TNN, GGA, all three field terms) for $\lambda$ from $\lambda_0/2$ to $3\lambda_0/2$, everything else fixed. The $\lambda$ values are those of the production sweep, `n_lam` points across $[\lambda_0/2, 3\lambda_0/2]$ merged with `n_lam` points across $[0, 0.4]$ eV, of which we keep the ones inside the window. $E_g$ is recomputed for each $\lambda$. Since $\eta$ is odd in $B$, $\chi = \eta(10\,{\rm T})/10$ T.

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper settings, with a quick setting in the comment. The cached results in `data/` have the resolution in their file names, so changing a line here starts a fresh run for that section.

In [ ]:
R_disk = 0.9       # disk radius (1/a)
h_lam = 0.002      # disk spacing (1/a); 0.005 for a quick look (0.01 gives wrong signs)
n_lam = 21         # points in each of the two lambda ranges; 5 for a quick look

## $\chi(\lambda)$

On the disks with $\nu = 5$ meV. About 15 s per value of $\lambda$ (25 to 33 values per material), 30 min in all at paper settings.

In [ ]:
lam_chi = {}
for m in materials:
    fn = f"data/lambda_{m}_R{R_disk}_h{h_lam}_n{n_lam}.npz"
    if not os.path.exists(fn):
        lam0 = lambda_SOC[m]
        lams = np.unique(np.round(np.concatenate([np.linspace(0.5 * lam0, 1.5 * lam0, n_lam), np.linspace(0, 0.4, n_lam)]), 6))
        lams = lams[(lams >= 0.5 * lam0 - 1e-9) & (lams <= 1.5 * lam0 + 1e-9)]
        grids = valley_grids(R_disk, h_lam)
        chi = []
        for lam in lams:
            eta = eta_mag(m, "TNN", 10.0, np.array([gap(m, lam=lam) + 0.1]), 0.005, grids, lam=lam)
            chi.append(eta[0, 0, 0, 0] / 10)
        np.savez(fn, lam=lams, chi=eta_SI(params(m)["a"]) * np.array(chi))
    lam_chi[m] = dict(np.load(fn))
    x, y = lam_chi[m]["lam"], lam_chi[m]["chi"]
    print(f"{m}: chi(lambda_0) = {1e4 * np.interp(lambda_SOC[m], x, y):+.3f} x 1e-4, linear slope {1e4 * np.polyfit(x, y, 1)[0]:+.2f} x 1e-4 per eV")

## Figure 12

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(TW, 1.85), gridspec_kw=dict(wspace=0.42))
for i, m in enumerate(materials):
    x, y = lam_chi[m]["lam"], lam_chi[m]["chi"]
    ax[i].plot(x, 1e4 * y, "o", color=colors[m], ms=2.6, mew=0)
    for deg, ls in ((1, "-"), (2, "--")):
        ax[i].plot(x, 1e4 * np.polyval(np.polyfit(x, y, deg), x), ls, color=colors[m], lw=0.8)
    ax[i].axvline(lambda_SOC[m], color="0.6", ls=":", lw=0.7)
    zero(ax[i])
    ax[i].set_xlabel(r"$\lambda$ (eV)")
    ax[i].set_title(names[m], pad=3, loc="right")
    label(ax[i], "abcd"[i])
    ax[i].xaxis.set_major_locator(MaxNLocator(4))
ax[0].set_ylabel(r"$\chi$ ($10^{-4}\,%s\,\mathrm{T^{-1}}$)" % UNIT)
save(fig, "lpge_lambda_grid")
plt.show()